Pre-processing data from Final_Fantasy_data.csv

In [2]:
# preprocess.py

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import joblib
import os
import logging

# Setup logging
os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    filename="logs/preprocess.log",
    level=logging.INFO,
    format="%(asctime)s:%(levelname)s:%(message)s"
)

def preprocess(csv_path, output_dir="data"):
    logging.info(f"Starting preprocessing for {csv_path}")

    df = pd.read_csv(csv_path)
    logging.info(f"Data loaded successfully. Shape: {df.shape}")
    print(f"Data loaded. Shape: {df.shape}")

    # Sanity check
    print("Unique home teams:", df['home_team'].unique())
    print("Unique away teams:", df['away_team'].unique())
    print("Unique batting innings:", df['batting_innings'].unique())

    # Label Encode player and venue
    player_encoder = LabelEncoder()
    venue_encoder = LabelEncoder()

    df['player_id'] = player_encoder.fit_transform(df['fullName'])
    df['venue_id'] = venue_encoder.fit_transform(df['venue'])
    logging.info("Player and Venue label encoding completed.")
    print(f"Player Encoder classes: {len(player_encoder.classes_)}")
    print(f"Venue Encoder classes: {len(venue_encoder.classes_)}")

    # Save encoders
    os.makedirs(output_dir, exist_ok=True)
    joblib.dump(player_encoder, os.path.join(output_dir, 'player_encoder.pkl'))
    joblib.dump(venue_encoder, os.path.join(output_dir, 'venue_encoder.pkl'))
    logging.info("Encoders saved.")

    # Only use home_team, away_team, batting_innings
    feature_cols = ['home_team', 'away_team', 'batting_innings']

    # Normalize text columns
    for col in ['home_team', 'away_team']:
        df[col] = df[col].str.strip().str.upper()  # Uppercase for consistency

    onehot_encoder = OneHotEncoder(sparse_output=False)
    onehot_features = onehot_encoder.fit_transform(df[feature_cols])
    print(f"One-hot encoded feature shape: {onehot_features.shape}")
    logging.info("One-hot encoding completed.")

    # Save one-hot encoder
    joblib.dump(onehot_encoder, os.path.join(output_dir, 'onehot_encoder.pkl'))

    # Create arrays
    player_ids = df['player_id'].values
    venue_ids = df['venue_id'].values
    target_fp = df['Total_FP'].values

    
    

    # Save arrays
    np.save(os.path.join(output_dir, 'player_ids.npy'), player_ids)
    np.save(os.path.join(output_dir, 'venue_ids.npy'), venue_ids)
    np.save(os.path.join(output_dir, 'onehot_inputs.npy'), onehot_features)
    np.save(os.path.join(output_dir, 'target_fp.npy'), target_fp)
    
    logging.info("Processed arrays saved successfully.")

    print("Preprocessing complete. Files saved.")

if __name__ == "__main__":
    preprocess(r"C:/Users/Preethi/Downloads/mlops_project/data/Final_Fantasy_data.csv")


Data loaded. Shape: (22362, 17)
Unique home teams: ['GT' 'LSG' 'CSK' 'RCB' 'MI' 'KKR' 'DC' 'PBKS' 'SRH' 'RR' 'RPS' 'GL' 'PWI'
 'Kochi']
Unique away teams: ['CSK' 'MI' 'GT' 'SRH' 'LSG' 'RR' 'RCB' 'PBKS' 'KKR' 'DC' 'RPS' 'GL' 'PWI'
 'Kochi']
Unique batting innings: [1 2]
Player Encoder classes: 705
Venue Encoder classes: 37
One-hot encoded feature shape: (22362, 30)
Preprocessing complete. Files saved.


In [ ]:
# train.py

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import os
import logging
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Setup logging
os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    filename="logs/train.log",
    level=logging.INFO,
    format="%(asctime)s:%(levelname)s:%(message)s"
)

def build_model(num_players, num_venues, onehot_dim, player_emb_dim=32, venue_emb_dim=8):
    logging.info("Building model architecture.")
    print(f"Building model with player_emb_dim={player_emb_dim}, venue_emb_dim={venue_emb_dim}, onehot_dim={onehot_dim}")

    # Inputs
    player_input = Input(shape=(1,), name='player_input')
    venue_input = Input(shape=(1,), name='venue_input')
    onehot_input = Input(shape=(onehot_dim,), name='onehot_input')

    # Embeddings
    player_embedded = Embedding(input_dim=num_players, output_dim=player_emb_dim)(player_input)
    venue_embedded = Embedding(input_dim=num_venues, output_dim=venue_emb_dim)(venue_input)

    # Flatten the embeddings
    player_vec = Flatten()(player_embedded)
    venue_vec = Flatten()(venue_embedded)

    # Concatenate all features (embeddings and one-hot)
    x = Concatenate()([player_vec, venue_vec, onehot_input])

    # Dense layers
    x = Dense(128, activation='relu')(x)
    x = Dense(64, activation='relu')(x)
    output = Dense(1, activation='linear')(x)  # Predict Total Fantasy Points

    model = Model(inputs=[player_input, venue_input, onehot_input], outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])  # Added MSE metric

    logging.info("Model compiled successfully.")
    return model

def train():
    logging.info("Starting training...")

    # Load processed data
    player_ids = np.load("processed_data/player_ids.npy")
    venue_ids = np.load("processed_data/venue_ids.npy")
    onehot_inputs = np.load("processed_data/onehot_inputs.npy")
    target_fp = np.load("processed_data/target_fp.npy")

    # Fix shape mismatch
    player_ids = player_ids.reshape(-1, 1)
    venue_ids = venue_ids.reshape(-1, 1)
    target_fp = target_fp.reshape(-1, 1)

    print(f"player_ids shape: {player_ids.shape}")
    print(f"venue_ids shape: {venue_ids.shape}")
    print(f"onehot_inputs shape: {onehot_inputs.shape}")
    print(f"target_fp shape: {target_fp.shape}")
    print(f"player_ids dtype: {player_ids.dtype}")
    print(f"venue_ids dtype: {venue_ids.dtype}")
    print(f"onehot_inputs dtype: {onehot_inputs.dtype}")
    
    # Split into train and validation sets (90-10 split)
    (player_train, player_val, 
     venue_train, venue_val, 
     onehot_train, onehot_val, 
     target_train, target_val) = train_test_split(
        player_ids, venue_ids, onehot_inputs, target_fp,
        test_size=0.1, random_state=42
    )
    
    num_players = int(player_ids.max()) + 1
    num_venues = int(venue_ids.max()) + 1
    onehot_dim = onehot_inputs.shape[1]
    print(f"Data summary - Players: {num_players}, Venues: {num_venues}, Onehot feature size: {onehot_dim}")

    # Build model
    model = build_model(num_players, num_venues, onehot_dim)

    # Train model
    history = model.fit(
        x=[player_train, venue_train, onehot_train],
        y=target_train,
        validation_data=([player_val, venue_val, onehot_val], target_val),
        epochs=50,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
        verbose=2
    )

    # Evaluate on validation set
    val_loss, val_mae, val_mse = model.evaluate(
        [player_val, venue_val, onehot_val], 
        target_val, 
        verbose=0
    )
    print(f"\nFinal Validation MSE: {val_mse:.4f}, MAE: {val_mae:.4f}")

    # Make predictions on validation set
    val_predictions = model.predict([player_val, venue_val, onehot_val])
    
    # Display sample predictions vs actual
    print("\nSample Validation Predictions:")
    print("Actual\tPredicted\tDifference")
    for i in range(10):  # Show first 10 samples
        actual = target_val[i][0]
        predicted = val_predictions[i][0]
        print(f"{actual:.1f}\t{predicted:.1f}\t\t{actual-predicted:.1f}")

    # Plot predictions vs actual
    plt.figure(figsize=(10, 6))
    plt.scatter(target_val, val_predictions, alpha=0.3)
    plt.plot([target_val.min(), target_val.max()], 
             [target_val.min(), target_val.max()], 'r--')
    plt.xlabel('Actual Fantasy Points')
    plt.ylabel('Predicted Fantasy Points')
    plt.title('Validation Set: Actual vs Predicted')
    plt.savefig('validation_predictions.png')
    plt.close()

    # Save model
    model.save("processed_data/fantasy_model.h5")
    print("Model saved at processed_data/fantasy_model.h5")

if __name__ == "__main__":
    train()

Error: 'original_data.csv' not found in 'processed_data' directory. Please ensure your original data is saved there.


In [17]:
import numpy as np
import pandas as pd # <-- Add pandas import
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import os
import logging
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Setup logging
os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    filename="logs/train.log",
    level=logging.INFO,
    format="%(asctime)s:%(levelname)s:%(message)s"
)

def build_model(num_players, num_venues, onehot_dim, player_emb_dim=32, venue_emb_dim=8):
    logging.info("Building model architecture.")
    print(f"Building model with player_emb_dim={player_emb_dim}, venue_emb_dim={venue_emb_dim}, onehot_dim={onehot_dim}")

    # Inputs
    player_input = Input(shape=(1,), name='player_input')
    venue_input = Input(shape=(1,), name='venue_input')
    onehot_input = Input(shape=(onehot_dim,), name='onehot_input')

    # Embeddings
    player_embedded = Embedding(input_dim=num_players, output_dim=player_emb_dim)(player_input)
    venue_embedded = Embedding(input_dim=num_venues, output_dim=venue_emb_dim)(venue_input)

    # Flatten the embeddings
    player_vec = Flatten()(player_embedded)
    venue_vec = Flatten()(venue_embedded)

    # Concatenate all features (embeddings and one-hot)
    x = Concatenate()([player_vec, venue_vec, onehot_input])

    # Dense layers
    x = Dense(128, activation='relu')(x)
    x = Dense(64, activation='relu')(x)
    output = Dense(1, activation='linear')(x)  # Predict Total Fantasy Points

    model = Model(inputs=[player_input, venue_input, onehot_input], outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])  # Added MSE metric

    logging.info("Model compiled successfully.")
    return model

def train():
    logging.info("Starting training...")

    # --- Load Original Context Data ---
    # !! Replace 'path/to/your/original_data.csv' with the actual path !!
    # !! Ensure this DataFrame's rows align with the .npy files before splitting !!
    try:
        original_df = pd.read_csv(r"C:\Users\arvin\Desktop\VSCode\DA5402- Machine Learning Operations\AI application\data\Final_Fantasy_data.csv")
        # Example: Selecting only necessary columns for context
        # !! Replace with your actual column names !!
        context_columns = ['fullName','venue','batting_innings','home_team', 'away_team'] # Add/remove columns as needed
        original_context_df = original_df[context_columns]
        print("Loaded original context data.")
    except FileNotFoundError:
        print("Error: Original context data file not found. Cannot display names.")
        print("Please provide the correct path to the original data CSV.")
        original_context_df = None # Set to None if loading fails
    except KeyError as e:
         print(f"Error: Column {e} not found in the original data CSV.")
         print("Please ensure the CSV contains the required context columns (e.g., 'player_name', 'venue_name').")
         original_context_df = None # Set to None if loading fails


    # Load processed data
    player_ids = np.load("processed_data/player_ids.npy")
    venue_ids = np.load("processed_data/venue_ids.npy")
    onehot_inputs = np.load("processed_data/onehot_inputs.npy")
    target_fp = np.load("processed_data/target_fp.npy")

    # --- Data Shape Validation ---
    if original_context_df is not None and len(original_context_df) != len(player_ids):
        print(f"Error: Mismatch in length between original context data ({len(original_context_df)}) and loaded features ({len(player_ids)}).")
        print("Ensure the original data corresponds exactly to the processed .npy files.")
        # Decide how to handle: exit, or proceed without context
        original_context_df = None # Disable context display due to mismatch

    # Fix shape mismatch for model input
    player_ids = player_ids.reshape(-1, 1)
    venue_ids = venue_ids.reshape(-1, 1)
    target_fp = target_fp.reshape(-1, 1)

    print(f"player_ids shape: {player_ids.shape}")
    print(f"venue_ids shape: {venue_ids.shape}")
    print(f"onehot_inputs shape: {onehot_inputs.shape}")
    print(f"target_fp shape: {target_fp.shape}")
    print(f"player_ids dtype: {player_ids.dtype}")
    print(f"venue_ids dtype: {venue_ids.dtype}")
    print(f"onehot_inputs dtype: {onehot_inputs.dtype}")

    # --- Split Data (Features and Context) ---
    # Use the same random_state to ensure alignment!
    split_args = {
        'test_size': 0.1,
        'random_state': 42
    }

    (player_train, player_val,
     venue_train, venue_val,
     onehot_train, onehot_val,
     target_train, target_val) = train_test_split(
        player_ids, venue_ids, onehot_inputs, target_fp, **split_args
    )

    # Split context data IF it was loaded successfully
    context_train_df, context_val_df = (None, None)
    if original_context_df is not None:
         context_train_df, context_val_df = train_test_split(
             original_context_df, **split_args
         )
         print("Context data split successfully.")


    num_players = int(np.max(player_ids)) + 1 # Use np.max for safety
    num_venues = int(np.max(venue_ids)) + 1   # Use np.max for safety
    onehot_dim = onehot_inputs.shape[1]
    print(f"Data summary - Players: {num_players}, Venues: {num_venues}, Onehot feature size: {onehot_dim}")

    # Build model
    model = build_model(num_players, num_venues, onehot_dim)

    # Train model
    history = model.fit(
        x=[player_train, venue_train, onehot_train],
        y=target_train,
        validation_data=([player_val, venue_val, onehot_val], target_val),
        epochs=50,
        batch_size=32,
        callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)],
        verbose=2
    )

    # Evaluate on validation set
    val_loss, val_mae, val_mse = model.evaluate(
        [player_val, venue_val, onehot_val],
        target_val,
        verbose=0
    )
    print(f"\nFinal Validation MSE: {val_mse:.4f}, MAE: {val_mae:.4f}")

    # Make predictions on validation set
    val_predictions = model.predict([player_val, venue_val, onehot_val])

    # --- Display sample predictions vs actual with context ---
    print("\nSample Validation Predictions:")
    # Print header based on whether context is available
    if context_val_df is not None:
        # Adjust the header to reflect the loaded columns
        # Using slightly more descriptive names and combining team/innings info
        header = "Player Name\tVenue\tMatch Details\tActual FP\tPredicted FP\tDifference"
        print(header)
        # Adjust separator length based on the new header
        print("-" * len(header.expandtabs()))
    else:
        # Fallback header if context isn't available
        print("Actual FP\tPredicted FP\tDifference")
        print("-------------------------------------") # Separator line

    for i in range(min(10, len(target_val))):  # Show up to 10 samples, handle smaller validation sets
        actual = target_val[i][0]
        predicted = val_predictions[i][0]
        diff = actual - predicted

        if context_val_df is not None:
            try:
                # Get context from the corresponding row in the validation context DataFrame
                context_row = context_val_df.iloc[i]

                # Extract data using the specific column names loaded earlier
                player_name = context_row.get('fullName', 'N/A')
                venue_name = context_row.get('venue', 'N/A')
                innings = context_row.get('batting_innings', '?')
                home_team = context_row.get('home_team', '?')
                away_team = context_row.get('away_team', '?')

                # Combine match-specific details into one string
                match_details = f"Inn:{innings} ({home_team} vs {away_team})"

                # Print the extracted context along with prediction results
                # Using tabs for separation - alignment might vary based on name lengths
                print(f"{player_name}\t{venue_name}\t{match_details}\t{actual:.1f}\t\t{predicted:.1f}\t\t{diff:+.1f}")

            except Exception as e:
                 # Fallback if accessing context fails for some reason
                 print(f"Error accessing context for index {i}: {e}")
                 # Print with placeholders matching the context header structure
                 print(f"N/A\tN/A\tN/A\t{actual:.1f}\t\t{predicted:.1f}\t\t{diff:+.1f}")

        else:
            # Print without context if not available
            print(f"{actual:.1f}\t\t{predicted:.1f}\t\t{diff:+.1f}")


    # Plot predictions vs actual
    plt.figure(figsize=(10, 6))
    plt.scatter(target_val, val_predictions, alpha=0.3)
    plt.plot([target_val.min(), target_val.max()],
             [target_val.min(), target_val.max()], 'r--')
    plt.xlabel('Actual Fantasy Points')
    plt.ylabel('Predicted Fantasy Points')
    plt.title('Validation Set: Actual vs Predicted')
    plt.savefig('validation_predictions.png')
    plt.close()
    print("\nValidation plot saved to validation_predictions.png")

    # Save model
    model_save_path = "processed_data/fantasy_model.h5"
    model.save(model_save_path)
    print(f"Model saved at {model_save_path}")

if __name__ == "__main__":
    train()

Loaded original context data.
player_ids shape: (22362, 1)
venue_ids shape: (22362, 1)
onehot_inputs shape: (22362, 30)
target_fp shape: (22362, 1)
player_ids dtype: int32
venue_ids dtype: int32
onehot_inputs dtype: float64
Context data split successfully.
Data summary - Players: 705, Venues: 37, Onehot feature size: 30
Building model with player_emb_dim=32, venue_emb_dim=8, onehot_dim=30
Epoch 1/50


C:\Users\arvin\AppData\Roaming\Python\Python310\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['player_input', 'venue_input', 'onehot_input']. Received: the structure of inputs=('*', '*', '*')
  warnings.warn(


629/629 - 2s - 3ms/step - loss: 1211.2058 - mae: 26.6888 - mse: 1211.2058 - val_loss: 997.2051 - val_mae: 25.1162 - val_mse: 997.2051
Epoch 2/50
629/629 - 1s - 2ms/step - loss: 1047.2473 - mae: 25.1757 - mse: 1047.2473 - val_loss: 989.0576 - val_mae: 24.7182 - val_mse: 989.0576
Epoch 3/50
629/629 - 1s - 2ms/step - loss: 1038.8330 - mae: 25.0333 - mse: 1038.8330 - val_loss: 991.6315 - val_mae: 24.8710 - val_mse: 991.6315
Epoch 4/50
629/629 - 1s - 2ms/step - loss: 1034.3647 - mae: 25.0017 - mse: 1034.3647 - val_loss: 992.1862 - val_mae: 24.5196 - val_mse: 992.1862
Epoch 5/50
629/629 - 1s - 2ms/step - loss: 1031.2599 - mae: 24.9446 - mse: 1031.2599 - val_loss: 985.3807 - val_mae: 24.7202 - val_mse: 985.3807
Epoch 6/50
629/629 - 1s - 2ms/step - loss: 1029.3778 - mae: 24.9084 - mse: 1029.3778 - val_loss: 992.5212 - val_mae: 24.8191 - val_mse: 992.5212
Epoch 7/50
629/629 - 1s - 2ms/step - loss: 1024.8677 - mae: 24.8778 - mse: 1024.8677 - val_loss: 1000.4259 - val_mae: 25.0944 - val_mse: 1000

In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
# import json <-- No longer needed
import joblib # <-- Add joblib
import os
import logging
# from sklearn.preprocessing import LabelEncoder, OneHotEncoder <-- Not strictly needed if just loading

# Setup logging
# logging.basicConfig(level=logging.INFO, format="%(asctime)s:%(levelname)s:%(message)s")

os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    filename="logs/model.log",
    level=logging.INFO,
    format="%(asctime)s:%(levelname)s:%(message)s"
)

# --- Configuration ---
ARTIFACTS_DIR = "processed_data"
MODEL_PATH = os.path.join(ARTIFACTS_DIR, "fantasy_model.h5")
# Paths to the saved sklearn encoder objects
PLAYER_ENCODER_PATH = os.path.join(ARTIFACTS_DIR, "player_encoder.pkl")
VENUE_ENCODER_PATH = os.path.join(ARTIFACTS_DIR, "venue_encoder.pkl")
ONEHOT_ENCODER_PATH = os.path.join(ARTIFACTS_DIR, "onehot_encoder.pkl")

# --- Load Artifacts ---
def load_artifacts():
    """Loads the trained model and necessary preprocessing artifacts (.pkl files)."""
    logging.info("Loading artifacts...")
    # Update required files to check for .pkl encoders
    required_files = [MODEL_PATH, PLAYER_ENCODER_PATH, VENUE_ENCODER_PATH, ONEHOT_ENCODER_PATH]
    if not all(os.path.exists(p) for p in required_files):
        logging.error("Error: One or more required artifact files are missing.")
        logging.error(f"Please ensure these files exist in '{ARTIFACTS_DIR}':")
        for p in required_files:
            if not os.path.exists(p):
                logging.error(f" - {os.path.basename(p)}")
        return None, None, None, None # Return None for all expected objects

    try:
        model = tf.keras.models.load_model(MODEL_PATH)
        logging.info(f"Model loaded successfully from {MODEL_PATH}")

        # Load encoders using joblib
        player_encoder = joblib.load(PLAYER_ENCODER_PATH)
        logging.info(f"Player encoder loaded from {PLAYER_ENCODER_PATH}")

        venue_encoder = joblib.load(VENUE_ENCODER_PATH)
        logging.info(f"Venue encoder loaded from {VENUE_ENCODER_PATH}")

        onehot_encoder = joblib.load(ONEHOT_ENCODER_PATH)
        logging.info(f"OneHot encoder loaded from {ONEHOT_ENCODER_PATH}")

        # Return the loaded objects
        return model, player_encoder, venue_encoder, onehot_encoder
    except Exception as e:
        logging.error(f"Error loading artifacts: {e}", exc_info=True)
        return None, None, None, None

# --- Preprocess Input ---
# Modify function signature to accept encoders
def preprocess_input(input_data, player_encoder, venue_encoder, onehot_encoder):
    """Converts raw input dictionary into the format required by the model using loaded encoders."""
    logging.info(f"Preprocessing input: {input_data}")

    # 1. Get Player ID using LabelEncoder
    player_name = input_data.get('fullName')
    if player_name is None:
        raise ValueError("Missing 'fullName' in input data.")
    try:
        # LabelEncoder expects an array/list and returns an array
        player_id = player_encoder.transform([player_name])[0]
    except ValueError:
        # Handle unseen player names during prediction
        known_players = player_encoder.classes_[:5] # Show first few known players
        logging.error(f"Player '{player_name}' was not seen during training.")
        logging.error(f"Known players start with: {', '.join(known_players)}...")
        # Option 1: Raise error (safer)
        raise ValueError(f"Unknown player: '{player_name}'. Cannot predict.")
        # Option 2: Assign a default ID (requires model understanding of this)
        # player_id = -1 # Or some other indicator
    player_id_input = np.array([[player_id]], dtype=np.int32)

    # 2. Get Venue ID using LabelEncoder
    venue_name = input_data.get('venue')
    if venue_name is None:
        raise ValueError("Missing 'venue' in input data.")
    try:
        venue_id = venue_encoder.transform([venue_name])[0]
    except ValueError:
        known_venues = venue_encoder.classes_[:5]
        logging.error(f"Venue '{venue_name}' was not seen during training.")
        logging.error(f"Known venues start with: {', '.join(known_venues)}...")
        raise ValueError(f"Unknown venue: '{venue_name}'. Cannot predict.")
    venue_id_input = np.array([[venue_id]], dtype=np.int32)

    # 3. Create One-Hot Encoded Features using OneHotEncoder
    # Define the order of features expected by the onehot_encoder
    # This should match the 'feature_cols' used in preprocess.py
    feature_cols_order = ['home_team', 'away_team', 'batting_innings']
    ohe_input_list = []
    try:
        for col in feature_cols_order:
            value = input_data.get(col)
            if value is None:
                raise ValueError(f"Missing feature '{col}' needed for one-hot encoding.")

            # Apply the SAME normalization as in preprocess.py
            if col in ['home_team', 'away_team']:
                value = str(value).strip().upper() # Ensure string, strip, and uppercase
            elif col == 'batting_innings':
                value = int(value) # Ensure integer type

            ohe_input_list.append(value)

        # OneHotEncoder expects a 2D array-like structure [[feature1, feature2, ...]]
        ohe_input_array = [ohe_input_list]

        # Transform using the loaded OneHotEncoder
        onehot_input = onehot_encoder.transform(ohe_input_array)
        # If sparse=False was used in training (as in your preprocess.py), it's already a dense array
        # If sparse=True, you might need onehot_input.toarray()

    except ValueError as e:
         # This can happen if a category (e.g., team name, innings number) wasn't seen during fit
         logging.error(f"Error transforming one-hot features: {e}", exc_info=True)
         logging.error("This might be due to an unknown category (e.g., team name) in the input.")
         raise ValueError(f"Failed to one-hot encode input features: {e}") from e
    except Exception as e:
        logging.error(f"Unexpected error during one-hot encoding: {e}", exc_info=True)
        raise ValueError("Unexpected error during one-hot encoding.") from e

    onehot_input = onehot_input.astype(np.float32) # Ensure correct dtype for the model

    logging.info(f"Preprocessing successful. Shapes: PlayerID {player_id_input.shape}, VenueID {venue_id_input.shape}, OneHot {onehot_input.shape}")
    # Note: No shape check against columns needed here, as onehot_encoder handles the output dimension
    return [player_id_input, venue_id_input, onehot_input]


# --- Make Prediction ---
# (This function remains the same as before)
def predict_fantasy_points(model, processed_input):
    """Makes a prediction using the loaded model and preprocessed input."""
    logging.info("Making prediction...")
    try:
        prediction = model.predict(processed_input)
        predicted_value = prediction[0][0]
        logging.info(f"Prediction successful: {predicted_value}")
        return predicted_value
    except Exception as e:
        logging.error(f"Error during model prediction: {e}", exc_info=True)
        raise


# --- Main Execution ---
if __name__ == "__main__":
    # Load artifacts using the updated function
    # Note the change in returned variables
    model, player_encoder, venue_encoder, onehot_encoder = load_artifacts()

    if model is None:
        print("\nExiting due to failure loading artifacts. Please check logs and file paths.")
    else:
        print("\n--- Fantasy Point Predictor ---")
        print("Enter match details (or leave blank to use example):")

        try:
            # --- Get User Input ---
            player_name_in = input(f"Enter Player Name (e.g., Virat Kohli): ")
            venue_name_in = input(f"Enter Venue Name (e.g., M. Chinnaswamy Stadium): ") # Use exact name from training
            innings_in = input(f"Enter Batting Innings (1 or 2): ")
            home_team_in = input(f"Enter Home Team (e.g., Royal Challengers Bengaluru): ")
            away_team_in = input(f"Enter Away Team (e.g., Chennai Super Kings): ")

            # --- Prepare Input Dictionary ---
            # Use defaults if user input is empty
            # IMPORTANT: Ensure examples match EXACTLY (case/spacing) how they appear in training data
            # before normalization, or use names guaranteed to be in the encoders.
            input_data = {
                "fullName": player_name_in or "Virat Kohli",
                "venue": venue_name_in or "M. Chinnaswamy Stadium", # Check exact name in venue_encoder.classes_ if unsure
                "batting_innings": innings_in or "1", # Will be converted to int in preprocess_input
                "home_team": home_team_in or "Royal Challengers Bengaluru", # Will be uppercased in preprocess_input
                "away_team": away_team_in or "Chennai Super Kings" # Will be uppercased in preprocess_input
            }

            print("\n--- Using Input Data (before internal normalization) ---")
            for key, value in input_data.items():
                print(f"- {key}: {value}")


            # Preprocess the raw input data using the loaded encoders
            # Note the change in arguments passed
            model_input = preprocess_input(
                input_data,
                player_encoder,
                venue_encoder,
                onehot_encoder
            )

            # Make the prediction using the model
            predicted_fp = predict_fantasy_points(model, model_input)

            print("\n--- Prediction Result ---")
            # Display the original (user-provided or default) names for better readability
            print(f"Predicted Fantasy Points for {input_data['fullName']} at {input_data['venue']}: {predicted_fp:.2f}")

        except ValueError as e:
            print(f"\nError: Invalid input or preprocessing failed - {e}")
        except Exception as e:
            print(f"\nAn unexpected error occurred: {e}")
            logging.error("Unexpected error in main execution block", exc_info=True)


Exiting due to failure loading artifacts. Please check logs and file paths.
